In [1]:
"""
二板策略 - RiceQuant Notebook格式
测试2021-2023年跨周期稳定性
"""

print("=" * 70)
print("二板策略 RiceQuant Notebook 测试")
print("=" * 70)

try:
    from datetime import datetime, timedelta
    import numpy as np

    print("\n步骤1: 获取交易日")
    test_dates = get_trading_dates("2022-04-01", "2022-04-30")
    print(f"2022年4月交易日: {len(test_dates)}天")

    print("\n步骤2: 统计涨停家数")
    results = []

    for i, date in enumerate(test_dates[:10]):  # 只测试前10天
        print(f"\n处理 {date.strftime('%Y-%m-%d')} ({i + 1}/10)")

        # 获取所有股票
        all_stocks = all_instruments("CS")
        stock_list = [
            s.order_book_id
            for s in all_stocks
            if not s.order_book_id.startswith(("688", "4", "8"))
        ][:300]

        # 统计涨停
        zt_count = 0
        zt_stocks = []

        for stock in stock_list[:100]:  # 测试前100只
            try:
                bars = history_bars(stock, 1, "1d", "close,limit_up", end_date=date)
                if bars is not None and len(bars) > 0:
                    if bars[0]["close"] >= bars[0]["limit_up"] * 0.99:
                        zt_count += 1
                        zt_stocks.append(stock)
            except:
                pass

        print(f"  涨停家数(样本100只): {zt_count}")

        if zt_count >= 5:  # 降低阈值，样本只有100只
            print(f"  涨停股票: {zt_stocks[:5]}")

            # 测试第一个涨停股票
            if len(zt_stocks) > 0:
                test_stock = zt_stocks[0]
                print(f"  测试股票: {test_stock}")

                try:
                    # 获取次日价格
                    next_bars = history_bars(
                        test_stock,
                        1,
                        "1d",
                        "open,close,limit_up",
                        end_date=test_dates[i + 1] if i + 1 < len(test_dates) else date,
                    )
                    if next_bars is not None and len(next_bars) > 0:
                        open_p = next_bars[0]["open"]
                        close_p = next_bars[0]["close"]
                        limit_p = next_bars[0]["limit_up"]

                        if open_p < limit_p * 0.99:  # 非涨停开盘
                            buy_p = open_p * 1.005
                            profit = (close_p / buy_p - 1) * 100
                            results.append(
                                {
                                    "date": date.strftime("%Y-%m-%d"),
                                    "stock": test_stock,
                                    "profit": profit,
                                }
                            )
                            print(f"  模拟买入，收益: {profit:.2f}%")
                except Exception as e:
                    print(f"  获取次日价格失败: {e}")

    print("\n" + "=" * 70)
    print("测试结果")
    print("=" * 70)

    if results:
        profits = [r["profit"] for r in results]
        wins = len([p for p in profits if p > 0])

        print(f"交易次数: {len(results)}")
        print(f"胜率: {wins / len(results) * 100:.1f}%")
        print(f"平均收益: {np.mean(profits):.2f}%")
        print(f"总收益: {sum(profits):.2f}%")

        print("\n交易详情:")
        for r in results:
            print(f"  {r['date']} {r['stock']}: {r['profit']:.2f}%")
    else:
        print("未找到符合条件的交易机会")

    print("\n" + "=" * 70)
    print("测试完成")
    print("=" * 70)

except Exception as e:
    print(f"\n错误: {e}")
    import traceback

    traceback.print_exc()


二板策略 RiceQuant Notebook 测试

步骤1: 获取交易日
2022年4月交易日: 19天

步骤2: 统计涨停家数

处理 2022-04-01 (1/10)



错误: 'str' object has no attribute 'order_book_id'


Traceback (most recent call last):
  File "<ipython-input-1-439b8753eaf9>", line 28, in <module>
    for s in all_stocks
  File "<ipython-input-1-439b8753eaf9>", line 29, in <listcomp>
    if not s.order_book_id.startswith(("688", "4", "8"))
AttributeError: 'str' object has no attribute 'order_book_id'
